In [ ]:
# --- Colab setup (auto-inserted; no-op outside Colab). tag: colab-bootstrap ---
import sys
if "google.colab" in sys.modules:
    import os, subprocess, pathlib
    _slug = "aniryou/full-stack-agentic-engineer"
    _root = pathlib.Path("/content") / "full-stack-agentic-engineer"
    if not _root.exists():
        subprocess.run(["git", "clone", "--depth", "1", f"https://github.com/{_slug}.git", str(_root)], check=True)
    os.chdir(_root / "07-application-agent-framework/retrieval-rag/embeddings-lab/exercises")
    for _c in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
        if (_c / "pyproject.toml").exists() or (_c / "setup.py").exists():
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(_c)]); break
        if (_c / "requirements.txt").exists():
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(_c / "requirements.txt")]); break
        if _c == _root:
            break
    if str(pathlib.Path.cwd()) not in sys.path:
        sys.path.insert(0, str(pathlib.Path.cwd()))


# Exercises 05 · Evaluation & fusion
The three functions every retrieval system owner ends up writing.
Solutions: `solutions/ex05_solutions.ipynb`.

In [ ]:
import numpy as np
from collections import Counter

## Task 1 — graded nDCG@k
`ndcg_at_k(gains, ideal_gains, k)` with `DCG = Σ gain_i / log2(i + 2)`.
Check: ranking with gains [3,2,0,1] vs ideal [3,2,1,0] → 0.98544.

In [ ]:
def dcg(gains, k):
    # ============ YOUR CODE HERE ============
    raise NotImplementedError("implement me, then re-run")

def ndcg_at_k(gains, ideal_gains, k=10):
    # ============ YOUR CODE HERE ============
    raise NotImplementedError("implement me, then re-run")

assert np.isclose(ndcg_at_k([3, 2, 0, 1], [3, 2, 1, 0], k=10), 0.98544, atol=1e-4)
assert np.isclose(ndcg_at_k([3, 2, 1, 0], [3, 2, 1, 0], k=10), 1.0)
print("ndcg_at_k ✓")

## Task 2 — Reciprocal Rank Fusion
`rrf(rankings, k)`: score(d) = Σ over rankings 1/(k + rank(d) + 1), rank
0-based; return doc ids sorted by score. Checks with k=1:
[a,b,c] + [c,a,b] → a: 1/2+1/3, c: 1/4+1/2, b: 1/3+1/4 → order a, c, b.
[a,b,c,d] + [b,c,d] → b: 1/3+1/2 = 5/6, c: 1/4+1/3 = 7/12, a: 1/2,
  d: 1/5+1/4 = 9/20 → order b, c, a, d. No two scores tie, so the order
  does not depend on how you break ties, and the constant shows:
  1/(k+rank) scores b 3/2, a 1, c 5/6, d 7/12 → b, a, c, d;
  1/(k+rank+2) scores b 7/12, c 9/20, d 11/30, a 1/3 → b, c, d, a.

In [ ]:
def rrf(rankings, k=60):
    # ============ YOUR CODE HERE ============
    raise NotImplementedError("implement me, then re-run")

assert rrf([["a", "b", "c"], ["c", "a", "b"]], k=1) == ["a", "c", "b"]
assert rrf([["a", "b", "c", "d"], ["b", "c", "d"]], k=1) == ["b", "c", "a", "d"], \
    "check the constant: score = 1/(k + rank + 1) with 0-based rank"
print("rrf ✓")

## Task 3 — the two halves of a BM25 term score
`bm25_idf(N, df) = log(1 + (N − df + 0.5)/(df + 0.5))` and
`bm25_tf_norm(f, dl, avgdl, k1, b) = f·(k1+1) / (f + k1·(1 − b + b·dl/avgdl))`.
The second is the part worth internalizing: term-frequency **saturates** and
long documents are **penalized**.

In [ ]:
def bm25_idf(N, df):
    # ============ YOUR CODE HERE ============
    raise NotImplementedError("implement me, then re-run")

def bm25_tf_norm(f, dl, avgdl, k1=1.5, b=0.75):
    # ============ YOUR CODE HERE ============
    raise NotImplementedError("implement me, then re-run")

assert np.isclose(bm25_idf(100, 10), np.log(1 + 90.5 / 10.5))
assert np.isclose(bm25_tf_norm(2, 100, 100), 10 / 7)
assert bm25_tf_norm(20, 100, 100) < 2.5 * bm25_tf_norm(1, 100, 100)   # saturation
assert bm25_tf_norm(2, 200, 100) < bm25_tf_norm(2, 50, 100)           # length penalty
print("bm25 components ✓")

## Task 4 (open) — break the hybrid
In notebook 05, replace RRF with a weighted score sum
`α·z(bm25) + (1−α)·z(dense)` (z = standardize scores per query). Sweep α.
Why does RRF usually win without tuning? (Hint: score scales vs rank scales.)